In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath("..")) # racine du dépôt bank-realtime-lakehouse
from generators.tx_generator import generate
from src.bank_lakehouse.bronze import ingest_bronze

In [0]:
generate(n=1000,files=2)

In [0]:
from src.bank_lakehouse.bronze import ingest_bronze

#Ingest data from "/Volumes/bank/ops/landing/transactions/" to bronze table bank.bronze.transactions_raw
ingest_bronze(spark).awaitTermination()



In [0]:
%sql

SELECT count(*) FROM bank.bronze.transactions_raw;

DESCRIBE EXTENDED bank.bronze.transactions_raw;  

DESCRIBE  HISTORY bank.bronze.transactions_raw; 

In [0]:
ingest_bronze(spark).awaitTermination()

In [0]:
%sql
--TEST IDEMPOTENCY ATER SECOND DEFAULT LOAD
SELECT * FROM bank.bronze.transactions_raw;
DESCRIBE HISTORY bank.bronze.transactions_raw;


ADD DATA TO FIRST LOAD 

In [0]:
from generators.tx_generator import generate
generate(n=200)

In [0]:
ingest_bronze(spark).awaitTermination() #ingest additonal 2OO row

In [0]:
%sql
Select count(*) from bank.bronze.transactions_raw; --> +200 rows

In [0]:

#SPRTIE VOLUME CHEICHPOINT
#/Volumes/bank/ops/checkpoints/>>tx_bronze/ 
#/offsets/ : 2 fichier 0, 1
#/commits/ : 2 fichier 0, 1
#/sources/ 1 dossier O , metata

Tester l’évolution de schéma

In [0]:
generate(n=100,with_headers=True)

In [0]:
from src.bank_lakehouse.bronze import ingest_bronze

# New `headers` column in landing files; table ACLs block mergeSchema, so evolve the table schema explicitly
existing_cols = {row["col_name"].strip().split(" ")[0] for row in spark.sql("DESCRIBE bank.bronze.transactions_raw").collect()}
if "headers" not in existing_cols:
    spark.sql("ALTER TABLE bank.bronze.transactions_raw ADD COLUMNS (headers STRUCT<schema_version: STRING, source: STRING>)")

ingest_bronze(spark).awaitTermination()

In [0]:
%sql
SELECT count(*) FROM bank.bronze.transactions_raw;
SELECT count_if(headers IS NOT NULL) AS avec_headers, count_if(headers IS NULL) AS sans_headers
FROM bank.bronze.transactions_raw;

In [0]:
generate(n=100,with_device_id=True)
# New `headers` column in landing files; table ACLs block mergeSchema,

In [0]:
ingest_bronze(spark).awaitTermination()

In [0]:
%sql
SELECT payload:device_id AS device_id
FROM bank.bronze.transactions_raw
WHERE payload:device_id IS NOT NULL;


CHECK RESCUE FLOW

In [0]:
SRC    = "/Volumes/bank/ops/landing/transactions_rescue/"
STATE  = "/Volumes/bank/ops/landing/_generator_state_rescue"
SCHEMA = "/Volumes/bank/ops/checkpoints/tx_rescue_schema"
CKPT   = "/Volumes/bank/ops/checkpoints/tx_rescue_bronze"
TARGET = "bank.bronze.tx_raw_rescue"

In [0]:
print(generate(n=200, landing=SRC, state_dir=STATE))

In [0]:
ingest_bronze(spark, source=SRC, schema_loc=SCHEMA, checkpoint=CKPT,
              target=TARGET, schema_evolution_mode="rescue").awaitTermination()

In [0]:
%sql
SELECT count(*) FROM bank.bronze.tx_raw_rescue;   -- 200 + doublons (voir "messages" du résumé)
DESCRIBE bank.bronze.tx_raw_rescue; 

In [0]:
generate(n=50, landing=SRC, state_dir=STATE, with_headers=True)

In [0]:
ingest_bronze(spark, source=SRC, schema_loc=SCHEMA, checkpoint=CKPT,
              target=TARGET, schema_evolution_mode="rescue").awaitTermination()

In [0]:
%sql
DESCRIBE bank.bronze.tx_raw_rescue;

SELECT count_if(_rescued_data IS NOT NULL) AS sauvees, count(*) AS total
FROM bank.bronze.tx_raw_rescue;

SELECT _rescued_data FROM bank.bronze.tx_raw_rescue
WHERE _rescued_data IS NOT NULL LIMIT 3;

SELECT _rescued_data:headers.source AS source, _rescued_data:_file_path AS fichier
FROM bank.bronze.tx_raw_rescue WHERE _rescued_data IS NOT NULL LIMIT 3;
